# AtmosAlert - Inteligência Espacial para Saúde Urbana

## Turma:1TSCOR 2025/2 
## Grupo: Data Insight
## Autores: 

|            Nome           |    RM    |
| ------------------------- | -------- |
| Daniela Pereira da Silva  | RM567079 |
| Fabio Ladislau do Amaral  | RM567292 |
| Regina Ferreira Bolsaneli | RM567828 |
| Thiago de Souza Perez     | RM568322 |
| Wagner Martins de Santana | RM567734 |

## Cenário
## Breve explicação sobre como queimadas geram fumaça que viaja até os centros urbanos, lotando hospitais. descrição do problema contextualizado com o uso de dados espaciais.

## Solução
### Como antecipar esse risco cruzando focos de calor espaciais com a direção dos ventos.

## ODSs Atendidas
### ODS 11 (Cidades Sustentáveis) e ODS 13 (Ação Contra a Mudança Global do Clima)

## Importação de Bibliotecas

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

## Origem e Dicionário de Dados
### Fonte 1: INPE / BDQueimadas (Focos de calor reais detectados por satélite).
### Fonte 2: NASA POWER (Velocidade e direção dos ventos a 50m).
### Fonte 3: Copernicus Sentinel-5P (Índice de Aerossóis/Fumaça).

Identificação da fonte de dados, carregamento e descrição das variáveis; 

## Ingestão de Dados

In [37]:
# Passando o caminho dos arquivos CSV
caminho_queimadas = '../data/raw/bdqueimadas_agosto_2023.csv'
caminho_fumaca = '../data/raw/fumaca_copernicus_agosto_2023.csv'
caminho_vento = '../data/raw/vento_nasa_agosto_2023.csv'

# Carregando os arquivos CSV em DataFrames
df_queimadas = pd.read_csv(caminho_queimadas)
df_fumaca = pd.read_csv(caminho_fumaca)
df_vento = pd.read_csv(caminho_vento)

# Validação do carregamento (linhas x colunas)
print(f"Queimadas (INPE): {df_queimadas.shape[0]} linhas e {df_queimadas.shape[1]} colunas.")
print(f"Fumaça (Copernicus): {df_fumaca.shape[0]} linhas e {df_fumaca.shape[1]} colunas.")
print(f"Vento (NASA): {df_vento.shape[0]} linhas e {df_vento.shape[1]} colunas.\n")

Queimadas (INPE): 5102 linhas e 10 colunas.
Fumaça (Copernicus): 798 linhas e 3 colunas.
Vento (NASA): 837 linhas e 4 colunas.



## Limpeza e Tratamento
### Texto explicando o rigor metodológico: tratamento de valores nulos (NaN), conversão de tipos de dados (datas em formato datetime), e padronização dos nomes das cidades e colunas.

In [ ]:
# Verificação de valores ausentes e tipo dos dados no dataframe de queimadas
info_df_queimadas = pd.DataFrame({
    "coluna":       df_queimadas.dtypes.index,
    "dtype":        df_queimadas.dtypes.values.astype(str),
    "non_null":     df_queimadas.count().values,
    "nulos":        df_queimadas.isnull().sum().values,
    "nulos_%":      (df_queimadas.isnull().mean().values * 100).round(2),
})
info_df_queimadas.style\
    .format({"nulos_%": "{:.2f}%"})\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

,coluna,dtype,non_null,nulos,nulos_%
0,id_bdq,int64,5102,0,0.00%
1,foco_id,str,5102,0,0.00%
2,lat,float64,5102,0,0.00%
3,lon,float64,5102,0,0.00%
4,datahora,datetime64[us],5102,0,0.00%
5,pais,str,5102,0,0.00%
6,estado,str,5102,0,0.00%
7,Municipio,str,5102,0,0.00%
8,bioma,str,5102,0,0.00%
9,Data,datetime64[us],5102,0,0.00%


In [47]:
# Verificação de valores ausentes e tipo dos dados no dataframe de ventos
info_df_vento = pd.DataFrame({
    "coluna":       df_vento.dtypes.index,
    "dtype":        df_vento.dtypes.values.astype(str),
    "non_null":     df_vento.count().values,
    "nulos":        df_vento.isnull().sum().values,
    "nulos_%":      (df_vento.isnull().mean().values * 100).round(2),
})
info_df_vento.style\
    .format({"nulos_%": "{:.2f}%"})\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

,coluna,dtype,non_null,nulos,nulos_%
0,Velocidade_Vento_50m,float64,837,0,0.00%
1,Direcao_Vento_50m,float64,837,0,0.00%
2,Data,datetime64[us],837,0,0.00%
3,Municipio,str,837,0,0.00%


In [48]:
# Verificação de valores ausentes e tipo dos dados no dataframe de fumaça
info_df_fumaca = pd.DataFrame({
    "coluna":       df_fumaca.dtypes.index,
    "dtype":        df_fumaca.dtypes.values.astype(str),
    "non_null":     df_fumaca.count().values,
    "nulos":        df_fumaca.isnull().sum().values,
    "nulos_%":      (df_fumaca.isnull().mean().values * 100).round(2),
})
info_df_fumaca.style\
    .format({"nulos_%": "{:.2f}%"})\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

,coluna,dtype,non_null,nulos,nulos_%
0,Data,datetime64[us],798,0,0.00%
1,Indice_Fumaca,float64,798,0,0.00%
2,Municipio,str,798,0,0.00%


In [ ]:
# Conversões de dados(string para data)
df_queimadas['Data'] = pd.to_datetime(df_queimadas['Data'])
df_queimadas['datahora'] = pd.to_datetime(df_queimadas['datahora'])
df_fumaca['Data'] = pd.to_datetime(df_fumaca['Data'])
df_vento['Data'] = pd.to_datetime(df_vento['Data'])

# Padronizando o nome da coluna de município para bater com as outras bases
df_queimadas.rename(columns={'municipio': 'Municipio'}, inplace=True)

# Estatistica Descritiva

In [41]:
# Estatística Descritiva do dataframe de queimadas e concatenção no eixo das colunas (axis=1).
# Preenchimento de NaN onde não se aplica.
desc_queimadas = pd.concat([df_queimadas.describe(), df_queimadas.describe(include=['object'])], axis=1)

desc_queimadas.style\
    .format(precision=3, na_rep="—")\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

/tmp/ipykernel_7011/2459310105.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  desc_queimadas = pd.concat([df_queimadas.describe(), df_queimadas.describe(include=['object'])], axis=1)


,id_bdq,lat,lon,datahora,Data,foco_id,pais,estado,Municipio,bioma
count,5102.000,5102.000,5102.000,5102,5102,5102,5102,5102,5102,5102
mean,1644998359.488,-9.814,-57.746,2023-08-18 12:01:15.135241,2023-08-17 18:16:13.735790,—,—,—,—,—
min,1641478304.000,-29.524,-71.665,2023-08-01 16:35:00,2023-08-01 00:00:00,—,—,—,—,—
25%,1643790720.250,-9.444,-65.087,2023-08-12 17:43:00,2023-08-12 00:00:00,—,—,—,—,—
50%,1645207448.000,-8.674,-55.254,2023-08-19 17:41:00,2023-08-19 00:00:00,—,—,—,—,—
75%,1646125076.750,-7.399,-53.222,2023-08-23 18:51:00,2023-08-23 00:00:00,—,—,—,—,—
max,1647733821.000,4.544,-36.577,2023-08-31 17:56:00,2023-08-31 00:00:00,—,—,—,—,—
std,1605253.318,5.327,8.036,—,—,—,—,—,—,—
unique,—,—,—,—,—,5102,1,26,26,5
top,—,—,—,—,—,22f7296b-bbec-3181-a8c1-39df740b6bff,Brasil,PARÁ,ALTAMIRA,Amazônia


In [42]:
# Estatística Descritiva do dataframe de fumaça e concatenção no eixo das colunas (axis=1).
# Preenchimento de NaN onde não se aplica.
desc_fumaca = pd.concat([df_fumaca.describe(), df_fumaca.describe(include=['object'])], axis=1)

desc_fumaca.style\
    .format(precision=3, na_rep="—")\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

/tmp/ipykernel_7011/4135019938.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  desc_fumaca = pd.concat([df_fumaca.describe(), df_fumaca.describe(include=['object'])], axis=1)


,Data,Indice_Fumaca,Municipio
count,798,798.000,798
mean,2023-08-15 00:36:05.413533,-0.286,—
min,2023-08-01 00:00:00,-1.962,—
25%,2023-08-08 00:00:00,-0.678,—
50%,2023-08-15 00:00:00,-0.244,—
75%,2023-08-22 00:00:00,0.116,—
max,2023-08-29 00:00:00,1.817,—
std,—,0.599,—
unique,—,—,27
top,—,—,SAO FRANCISCO DE PAULA


In [43]:
# Estatística Descritiva do dataframe de vento e concatenção no eixo das colunas (axis=1).
# Preenchimento de NaN onde não se aplica.
desc_vento = pd.concat([df_vento.describe(), df_vento.describe(include=['object'])], axis=1)

desc_vento.style\
    .format(precision=3, na_rep="—")\
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])

/tmp/ipykernel_7011/560714571.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  desc_vento = pd.concat([df_vento.describe(), df_vento.describe(include=['object'])], axis=1)


,Velocidade_Vento_50m,Direcao_Vento_50m,Data,Municipio
count,837.000,837.000,837,837
mean,4.541,121.926,2023-08-16 00:00:00,—
min,0.940,0.000,2023-08-01 00:00:00,—
25%,2.680,67.100,2023-08-08 00:00:00,—
50%,4.230,100.200,2023-08-16 00:00:00,—
75%,6.250,133.900,2023-08-24 00:00:00,—
max,10.550,359.400,2023-08-31 00:00:00,—
std,2.146,90.167,—,—
unique,—,—,—,27
top,—,—,—,FEIJO


## Junção dos DataFrames

In [55]:
# Agrupamento dos focos do INPE
# Conta a quantidade de focos (linhas) para cada combinação de dia e cidade
df_queimadas_agrupado = (df_queimadas
                         .groupby(['Data', 'Municipio'])
                         .size()
                         .reset_index(name='Focos_Calor'))

# União das duas tabelas que já possuem registos diários (Fumaça e Vento)
df_consolidado = pd.merge(df_fumaca, df_vento, on=['Data', 'Municipio'], how='inner')

# Adição dos focos de calor agrupados usando 'left' 
df_atmosalert = pd.merge(df_consolidado, df_queimadas_agrupado, on=['Data', 'Municipio'], how='left')

# Se a união gerou valores nulos (NaN) em Focos_Calor, significa que não houve fogo naquele dia.
# Substituímos por 0 e convertemos para número inteiro.
df_atmosalert['Focos_Calor'] = df_atmosalert['Focos_Calor'].fillna(0).astype(int)

# Ordena o DataFrame final por Município e Data para manter a linha do tempo organizada
df_atmosalert = df_atmosalert.sort_values(by=['Municipio', 'Data']).reset_index(drop=True)

print(f"Total de registos gerados: {df_atmosalert.shape[0]} linhas.")

# Validação da estrutura final
print("Estrutura do novo DataFrame unificado:")
#df_atmosalert.info()
display(df_atmosalert.head(10))

Total de registos gerados: 798 linhas.
Estrutura do novo DataFrame unificado:


,Data,Indice_Fumaca,Municipio,Velocidade_Vento_50m,Direcao_Vento_50m,Focos_Calor
0,2023-08-01,-1.077591,ALTAMIRA,2.54,50.0,0
1,2023-08-02,-0.678123,ALTAMIRA,2.46,59.4,31
2,2023-08-03,-0.107790,ALTAMIRA,2.50,60.9,6
3,2023-08-04,-0.501449,ALTAMIRA,2.46,57.8,19
4,2023-08-05,-1.313093,ALTAMIRA,2.66,60.9,14
5,2023-08-06,-0.825194,ALTAMIRA,2.90,68.8,6
6,2023-08-07,-0.592765,ALTAMIRA,2.82,58.2,33
7,2023-08-08,0.032284,ALTAMIRA,2.60,62.8,6
8,2023-08-09,-0.715192,ALTAMIRA,2.73,51.9,86
9,2023-08-10,-0.793383,ALTAMIRA,2.96,58.6,2


## Visualizações Iniciais

In [ ]:
## Lorem Ipsum

## Teste de Hipótese

In [ ]:
## Lorem Ipsum

## Correlação
### Texto analítico com conclusões, limitações e contribuição para o Global Solution. 

In [ ]:
## Lorem Ipsum

## Conclusões e Impacto para Tomada de Decisão

In [1]:
## Lorem Ipsum

## Link do protótipo web
·	Inserir aqui o link do protótipo web.